In [25]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS

In [26]:
load_dotenv()

True

In [28]:
groq_key=os.getenv("GROQ_API_KEY")
jina_key=os.getenv("JINA_API_KEY")

### Loading Data

In [4]:
DATA_FILE_PATH=os.path.join("data","hr_policy.txt")

### Data Ingestion

In [5]:
text_loader=TextLoader(DATA_FILE_PATH,encoding="utf-8")

documents=text_loader.load()

print("DATA LOADED")

print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

### Langchain Document

In [6]:
len(documents)

1

In [7]:
print(f"Total characters in the document : {len(documents[0].page_content)}")

Total characters in the document : 2598


In [8]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,
                               chunk_overlap=50)

chunks=text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [9]:
len(chunks)

9

In [10]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [11]:
print(chunks[1])

page_content='1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.' metadata={'source': 'data\\hr_policy.txt'}


In [12]:
print(chunks[6])

page_content='6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.' metadata={'source': 'data\\hr_policy.txt'}


### Embedding the data

In [15]:
embedding_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print(f"Embedding model : {embedding_model}")

Embedding model : session=<requests.sessions.Session object at 0x000001E2915EB910> model_name='jina-embeddings-v2-base-en' jina_api_key=None


### Vector stores

In [17]:
Vector_store=FAISS.from_documents(chunks,embedding_model)
print("Total vectors : ",Vector_store.index.ntotal)

Total vectors :  9


In [19]:
test_query="How many month probation is there?"

test_results=Vector_store.similarity_search(test_query,k=2)

print(f"Query: {test_query}")
for i,match in enumerate(test_results,start=1):
    print(f"{i} = {match.page_content}")
    print()
   

Query: How many month probation is there?
1 = 3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Performance is reviewed at the end of the probation period to confirm employment.

2 = 4. NOTICE PERIOD
Employees who wish to resign must serve a notice period of 30 days.
During the probation period, the notice period is reduced to 15 days.
The company may waive the notice period at its discretion, with full and final settlement
processed within 45 days of the last working day.



### LLM

In [29]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E2FCE13590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E2AAED3350>, model_name='openai/gpt-oss-120b', temperature=0.5, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [31]:
response=llm.invoke("what is rag answer in 1 line?")
print(response.content)

RAG (Retrieval‑Augmented Generation) merges external document retrieval with a language model’s generation to produce more accurate, up‑to‑date answers.
